# Лабораторная работа № 1.2
## *Извлечение признаков из текста*
по курсу Технологии обработки естественного языка  
**направление:** Речевые технологии и машинное обучение  
**преподаватель:** Коротеева Олеся В.  
**выполнил:** Янкин Иван Ю.  
**группа:** М4121

In [5]:
import numpy as np
import pandas as pd
import pymorphy3
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')
from num2words import num2words
from tqdm.auto import tqdm
tqdm.pandas()
from functools import lru_cache
from gensim.models import Word2Vec, KeyedVectors, FastText
from gensim.models.fasttext import FastTextKeyedVectors
from ufal.udpipe import Model, Pipeline
import multiprocessing
import zipfile
cores = multiprocessing.cpu_count()

custom_exceptions = {'нет', 'не', 'опять', 'разве', 'хорошо',
                     'ни', 'ничего',  'наконец',
                     'совсем', 'уже', 'очень', 'чересчур',
                     'слишком', 'никогда'}
stop_words = set(stopwords.words('russian')) - custom_exceptions

import string
punctuations = list(string.punctuation) + ['..', '...', '–', '—', '«', '»', '“', '”']

morph = pymorphy3.MorphAnalyzer()
@lru_cache(maxsize=10000)
def get_morph(word):
    return morph.parse(word)[0]

from pathlib import Path
Path("./features").mkdir(parents=True, exist_ok=True)
Path("./w2v_data").mkdir(parents=True, exist_ok=True)
Path("./fasttext_data").mkdir(parents=True, exist_ok=True)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vanya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vanya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## загрузка датасета

In [14]:
df = pd.read_csv('./datasets/prepared_dataset.csv')
text_column_prepared = df.ttext
df.head(3)

,id,tdate,tname,ttext,ttype,trep,trtw,tfav,tstcount,tfoll,tfrien,listcount
0,408906762813579264,1386325944,dugarchikbellko,"['работа', 'полный', 'пиддес', 'каждый', 'закр...",-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"['коллега', 'сидеть', 'рубиться', 'из-за', 'до...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,"['говорить', 'обещаной', 'год', 'ждать']",-1,0,0,0,718,49,249,0


In [9]:
def get_avg_vector(tokens, model):
    wv = getattr(model, 'wv', model)
    
    vectors = [wv[token] for token in tokens if token in wv]
    if not vectors:
        return np.zeros(wv.vector_size)
    
    return np.mean(vectors, axis=0)

## извлечение Word2Vec векторов

### 1. обучение кастомных векторов

In [10]:
w2v_custom = Word2Vec(sentences=text_column_prepared.tolist(), vector_size=300, window=5, min_count=2, workers=cores)

In [11]:
w2v_features = text_column_prepared.progress_apply(lambda text_tokens: get_avg_vector(text_tokens, w2v_custom))
df_w2v_custom = pd.DataFrame(w2v_features.tolist())
df_w2v_custom.to_csv('./features/w2v_features.csv', index=False)
df_w2v_custom.head(3)

100%|██████████| 226834/226834 [00:13<00:00, 17378.16it/s]


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-0.008142,0.490481,-0.033060,0.498531,-0.055855,0.037159,0.152650,0.756350,-0.125811,-0.225562,...,0.046635,0.583036,0.169590,0.046353,0.624326,0.400729,-0.068106,0.068823,0.317310,-0.283969
1,-0.225103,0.389925,0.035950,0.429879,-0.143852,-0.132950,0.194908,0.486936,-0.104255,-0.273292,...,0.200268,0.155855,0.323278,-0.121441,0.143238,0.181186,0.093045,-0.119584,0.397214,-0.042742
2,-0.078603,0.776275,-0.055454,-0.001217,0.414562,-0.168990,0.202837,1.109150,0.380498,-0.033852,...,-0.655202,1.013461,0.568893,0.710796,1.136848,0.950809,0.133678,0.048560,0.870017,-0.646362


### 2. использование предобученных векторов
- ruscorpora_upos_cbow_300_20_2019

#### скачаем модель

In [12]:
%%capture
# !curl.exe -L -o ./w2v_data/ruscorpus.zip https://vectors.nlpl.eu/repository/20/180.zip

In [13]:
with zipfile.ZipFile('./w2v_data/ruscorpus.zip', 'r') as archive:
    archive.extract('model.bin', path='./w2v_data/')

FileNotFoundError: [Errno 2] No such file or directory: './w2v_data/ruscorpus.zip'

In [ ]:
w2v_pretrained = KeyedVectors.load_word2vec_format('./w2v_data/model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)

#### предобработаем тексты

In [ ]:
%%capture
# !curl -L -o ./w2v_data/rus_preprocessing_udpipe.py https://raw.githubusercontent.com/akutuzov/webvectors/refs/heads/master/preprocessing/rus_preprocessing_udpipe.py
# !curl -L -o ./w2v_data/udpipe_syntagrus.model https://rusvectores.org/static/models/udpipe_syntagrus.model

In [ ]:
from w2v_data.rus_preprocessing_udpipe import num_replace, clean_token, clean_lemma, list_replace, unify_sym, process

model = Model.load('./w2v_data/udpipe_syntagrus.model')
process_pipeline = Pipeline(
    model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu"
)

def process_text(text):
    res = unify_sym(text.strip())
    output = process(process_pipeline, text=res)
    return output
    
text_column_prepared_w2v = text_column.progress_apply(process_text)

100%|█████████████████████████████████████████████████████████████████████████| 226834/226834 [28:48<00:00, 131.26it/s]


In [ ]:
w2v_pretrained_features = text_column_prepared_w2v.progress_apply(lambda text_tokens: get_avg_vector(text_tokens, w2v_pretrained))
df_w2v_pretrained = pd.DataFrame(w2v_pretrained_features.tolist())
df_w2v_pretrained.to_csv('./features/w2v_pretrained_features.csv', index=False)
df_w2v_pretrained.head(3)

100%|███████████████████████████████████████████████████████████████████████| 226834/226834 [00:06<00:00, 36944.74it/s]


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.533661,-0.722712,1.447201,0.864545,-0.250062,0.512452,-1.482361,0.830283,-0.917641,-0.571312,...,-1.483107,-1.027939,-0.928173,1.128045,-0.069840,0.629521,-0.571891,0.427526,1.721985,0.335157
1,-0.014240,0.215863,-0.266485,0.726037,0.194412,-0.699286,0.035254,0.129613,-0.428507,0.444191,...,-0.769201,-0.179307,-0.578540,-0.537741,-0.420191,0.340457,-0.724334,0.608848,0.228702,0.211010
2,-0.012523,-1.649494,-0.708066,1.450534,-0.423977,-1.056427,-1.641925,0.269994,-1.368538,-1.480053,...,-0.648409,-0.984681,-1.133022,0.359423,-0.700835,1.320711,1.271321,-0.269154,-0.198932,-0.412483


## извлечение FastText векторов

### 3. обучение кастомных векторов

In [ ]:
fasttext_custom = FastText(vector_size=300, window=5, min_count=2, workers=cores)
fasttext_custom.build_vocab(corpus_iterable = text_column_prepared)
fasttext_custom.train(text_column_prepared, total_examples=fasttext_custom.corpus_count, epochs=10)

(13129719, 15683660)

In [ ]:
fasttext_features = text_column_prepared.progress_apply(lambda text_tokens: get_avg_vector(text_tokens, fasttext_custom))
df_fasttext_custom = pd.DataFrame(fasttext_features.tolist())
df_fasttext_custom.to_csv('./features/fasttext_features.csv', index=False)
df_fasttext_custom.head(3)

100%|███████████████████████████████████████████████████████████████████████| 226834/226834 [00:12<00:00, 18643.09it/s]


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.283502,0.162135,-0.606517,-0.326450,-0.056003,-0.225850,0.268860,0.196408,0.304186,0.126606,...,0.349714,-0.210757,-0.311463,-0.071789,-0.128767,0.526589,0.361547,0.540773,-0.267095,-0.228504
1,0.429516,-0.009054,-0.617236,-0.294672,-0.479848,0.230174,0.043504,-0.250544,-0.343393,0.022759,...,0.046997,-0.087383,0.931426,-0.086072,0.424711,1.035743,0.299990,0.051693,-0.325206,-0.155958
2,0.168808,0.459507,0.887384,-0.298448,0.776057,-0.565728,-0.147707,0.003716,0.608935,0.670147,...,0.048393,-0.500992,-1.517661,-0.186154,-0.325305,0.165189,-1.005675,0.252811,-0.197574,-0.535904


### 4. использование предобученных векторов
- eowac_lemmas_none_fasttextskipgram_300_5_2020

#### скачаем модель

In [ ]:
%%capture
# !curl.exe -L -o ./fasttext_data/fasttext_corpus.zip https://vectors.nlpl.eu/repository/20/213.zip

In [ ]:
with zipfile.ZipFile('./fasttext_data/fasttext_corpus.zip', 'r') as archive:
    archive.extractall(path='./fasttext_data/')

In [ ]:
fasttext_pretrained = FastTextKeyedVectors.load('./fasttext_data/model.model')
fasttext_features_pretrained = text_column_prepared.progress_apply(lambda text_tokens: get_avg_vector(text_tokens, fasttext_pretrained))
df_fasttext_pretrained = pd.DataFrame(fasttext_features_pretrained.tolist())
df_fasttext_pretrained.to_csv('./features/fasttext_features_pretrained.csv', index=False)
df_fasttext_pretrained.head(3)

100%|███████████████████████████████████████████████████████████████████████| 226834/226834 [00:14<00:00, 16101.68it/s]


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,-0.020151,0.069581,-0.053569,0.247671,0.137248,0.200066,-0.015692,0.134414,0.054312,0.103591,...,0.026138,0.147686,0.087791,-0.046578,-0.166233,0.080708,0.068734,-0.109995,-0.293939,-0.004935
1,0.127168,0.089284,-0.036961,0.155164,0.107712,-0.013820,0.104346,-0.052077,0.089333,0.015305,...,0.183256,0.116775,0.120266,0.002584,-0.170446,0.070587,0.060320,-0.085640,-0.179319,-0.072816
2,-0.063084,-0.015199,-0.170328,0.011000,0.212626,0.262083,-0.008568,0.071123,-0.030609,-0.008856,...,0.061179,-0.031331,0.097060,0.142485,-0.079269,-0.090395,0.149548,0.015660,-0.199339,0.008175
